# Geocoding CT Stroke Centers

In this notebook, we will obtain the latitude and longitude of each of the stroke centers in the state of Connecticut. The list of stroke centers is obtained from the following link: https://portal.ct.gov/dph/emergency-medical-services/ems/certified-stroke-centers?language=en_US. We use the following designations:

- Basic stroke care: Acute or Primary Stroke Center
- Advanced stroke care: Thrombectomy-capable or Comprehensive Stroke Center

## Creating a Dataframe of All Stroke Centers in CT

In [4]:
import pandas as pd

In [162]:
ct_stroke = pd.DataFrame({

    "name": [

        # Basic stroke centers
        
        "Bridgeport Hospital Milford Campus",
        "Charlotte Hungerford Hospital",
        "Day Kimball Hospital",
        "Greenwich Hospital",
        "Griffin Hospital",
        "THOCC - New Britain General",
        "THOCC - Bradley General",
        "Johnson Memorial Hospital",
        "Lawrence + Memorial Hospital",
        "Manchester Memorial Hospital",
        "Middlesex Hospital",
        "MidState Hospital",
        "New Milford Hospital",
        "Rockville General Hospital",
        "Sharon Hospital",
        "St. Mary's Hospital",
        "Stamford Hospital",
        "John Dempsey Hospital, Farmington, CT",
        "Waterbury Hospital",
        "Windham Hospital",
        "William W. Backus Hospital",
        "Yale New Haven Hospital Saint Raphael Campus, New Haven, CT",

        # Advanced stroke centers
        
        "Bridgeport Hospital",
        "Danbury Hospital",
        "Hartford Hospital",
        "Norwalk Hospital",
        "Saint Francis Hospital and Medical Center",
        "St. Vincent's Medical Center",
        "Yale New Haven Hospital"

    ],

    "group": [

        # Basic stroke centers
        
        "Basic",
        "Basic",
        "Basic",
        "Basic",
        "Basic",
        "Basic",
        "Basic",
        "Basic",
        "Basic",
        "Basic",
        "Basic",
        "Basic",
        "Basic",
        "Basic",
        "Basic",
        "Basic",
        "Basic",
        "Basic",
        "Basic",
        "Basic",
        "Basic",
        "Basic",

        # Advanced stroke centers
        
        "Advanced",
        "Advanced",
        "Advanced",
        "Advanced",
        "Advanced",
        "Advanced",
        "Advanced"
    ]
})

## Obtaining Coordinates

In [165]:
from geopy.geocoders import Nominatim
import pandas as pd
import time

In [167]:
geolocator = Nominatim(
    user_agent="stroke_project",
    timeout=10
)

#### Manually Entering Latitude and Longitude for Hospitals that Cannot be Found Using Nominatim

In [170]:
manual_coords = {
    "THOCC - New Britain General": (41.6676, -72.7793),
    "THOCC - Bradley General": (41.5954, -72.8786),
    "MidState Hospital": (41.5584, -72.7963)
}

### Defining a Function to Obtain the Coordinates

In [173]:
def get_coordinates(hospital_name):

    if hospital_name in manual_coords:
        lat, lon = manual_coords[hospital_name]
        return pd.Series([lat, lon])

    try:

        time.sleep(1)

        location = geolocator.geocode(
            hospital_name + ", Connecticut"
        )

        if location:
            return pd.Series([
                location.latitude,
                location.longitude
            ])

        return pd.Series([None, None])

    except:

        return pd.Series([
            None,
            None
        ])


### Calling the Function to Obtain the Latitude and Longitude

In [176]:
ct_stroke[
    ["latitude", "longitude"]
] = (
    ct_stroke["name"]
    .apply(get_coordinates)
)

In [177]:
ct_stroke.head()

,name,group,latitude,longitude
0,Bridgeport Hospital Milford Campus,Basic,41.216545,-73.065360
1,Charlotte Hungerford Hospital,Basic,41.792271,-73.133769
2,Day Kimball Hospital,Basic,41.906093,-71.913028
3,Greenwich Hospital,Basic,41.034299,-73.630646
4,Griffin Hospital,Basic,41.336435,-73.090512


## Creating Separate Datasets for Basic and Advanced Stroke Care

### Basic Stroke Centers

In [180]:
basic_ct_stroke = (
    ct_stroke[
        ct_stroke["group"] == "Basic"
    ]
)

### Advanced Stroke Centers

In [182]:
advanced_ct_stroke = (
    ct_stroke[
        ct_stroke["group"] == "Advanced"
    ]
)

## Saving to CSV File

In [190]:
ct_stroke.to_csv(
    "ct_stroke_centers_geocoded.csv",
    index=False
)

In [192]:
basic_ct_stroke.to_csv("ct_basic_geocoded.csv",
                       index = False
                      )

In [194]:
advanced_ct_stroke.to_csv("ct_advanced_geocoded.csv",
                          index = False
                         )